# Gradient boosting do scikit-learn: linha extra ao lado do XGBoost

O slide 7 pede o gradient boosting "usado para comparação", e o slide 28 lista os parâmetros dele.
Este notebook mede o `GradientBoostingClassifier` no placar comum (`docs/pre-proposta.md`, "Placar comum"), no mesmo fluxo do XGBoost:

1. **Trial 0:** os padrões do scikit-learn, no placar.
2. **Busca:** Optuna nas 5 dobras oficiais, com o procedimento do placar em cada dobra (`comum.estudo_placar`).
3. **Melhor trial no placar:** medido de novo nas 15 divisões do placar, que a busca não viu.
4. **Dados:** sem oversampling e com as três features do `docs/GUIA_DADOS.md`, contra o melhor trial.
5. **Final:** só troca o melhor trial por uma variante que **ganhou** dele. A linha `("gradient_boosting", "final")` recebe o `vs_xgboost` sozinha.
6. **Teste:** uma vez, com `REVELAR_TESTE = True`, depois de tudo congelado.

Em cada dobra, 80% do treino ajustam e 20% decidem o número de árvores; depois o modelo retreina no treino inteiro da dobra com esse número.
A dobra de validação só mede.

O espaço de busca está em `modelo.py`, com a medida que justifica cada escolha.
O `criterion` do slide 28 ficou de fora: no scikit-learn 1.9.1 ele está obsoleto e não muda o ajuste.

## Como executar

Na CPU, localmente, a partir de qualquer pasta do repositório, numa linha só:

`jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=-1 experimentos/gradient_boosting/gradient_boosting.ipynb`

O CSV é procurado em `CHURN_CSV`, em `dados/telco_ibm.csv`, no Input do Kaggle e, por fim, baixado pelo `kagglehub`.
O estudo fica em `resultados/gradient_boosting/gradient_boosting.db` e retoma de onde parou.

O scikit-learn treina uma árvore de cada vez, num núcleo só.
Por isso as dobras rodam em paralelo, em `THREADS` threads: medido numa dobra real, 4 dobras juntas levaram 27 s, contra 76 s uma depois da outra, com o mesmo número de árvores.
O resultado não depende de `THREADS`; os testes conferem que o estudo e o placar saem iguais aos da execução sequencial.

In [1]:
# ===== Configuração da execução (edite aqui) =====
N_TRIALS = 100          # mesmo N para todos os modelos (pré-proposta)
N_SEMENTES = 5          # sementes da avaliação final no teste
THREADS = 5             # dobras em paralelo; não muda o resultado, só o tempo
REVELAR_TESTE = True    # True só com a configuração congelada: o teste é usado uma única vez
MODO_TESTE = False      # True = passada rápida para validar o notebook, em uma pasta descartável

In [2]:
import json, os, sys, time
from pathlib import Path

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "comum.py").exists())
sys.path[:0] = [str(REPO), str(REPO / "experimentos" / "gradient_boosting")]
import fluxo

os.environ["CHURN_CSV"] = fluxo.achar_csv(os.environ.get("CHURN_CSV", ""))
if MODO_TESTE:
    os.environ["CHURN_RESULTADOS"] = str(REPO / "execucoes" / time.strftime("teste-gradient_boosting-%Y%m%d-%H%M"))
    N_TRIALS, N_SEMENTES = 3, 2

import numpy as np, optuna, pandas as pd, sklearn
from scipy.stats import ks_2samp
import comum, modelo as M

if MODO_TESTE:
    M.PADRAO = dict(M.PADRAO, learning_rate=0.3)   # menos árvores na conferência; a passada de verdade usa o padrão

PASTA = comum.RESULTADOS
PASTA_ESTUDO = os.path.join(PASTA, M.MODELO)
os.makedirs(PASTA_ESTUDO, exist_ok=True)
pd.set_option("display.max_columns", None, "display.width", 200)
print("scikit-learn", sklearn.__version__, "| optuna", optuna.__version__, "| CSV:", os.environ["CHURN_CSV"])
print("placar em", PASTA, "|", THREADS, "threads")

scikit-learn 1.9.1 | optuna 5.0.0 | CSV: C:\Users\rafae\.cache\kagglehub\datasets\kapturovalexander\customers-churned-in-telecom-services\versions\3\customer_churn_telecom_services.csv
placar em C:\dev\Faculdade\churn_project\resultados | 5 threads


In [3]:
# Verificações rápidas antes de gastar tempo de treino
X_tr, X_te, y_tr, y_te = comum.dividir(comum.carregar_df())
assert comum.assinatura_teste() == "e90ba6fab761f037", "o teste mudou: os números não seriam comparáveis"
if MODO_TESTE:  # o teste de verdade nunca entra numa passada de conferência: um pedaço do treino faz o papel dele
    X_te, y_te = X_tr.iloc[:500], y_tr.iloc[:500]
ASS = comum.assinatura_treino(X_tr)
rng = np.random.default_rng(1); y_ = (rng.random(2000) < .265).astype(int); p_ = np.clip(rng.normal(.3 + .25 * y_, .2), 0, 1)
assert abs(comum.ks(y_, p_) - ks_2samp(p_[y_ == 1], p_[y_ == 0]).statistic) < 1e-9
print(f"{len(X_tr)} clientes no treino | assinatura do treino {ASS} | KS conferido contra scipy")


def config(hp, features=(), over=True):
    return dict(hp=dict(hp), features=list(features), over=over)


def medir(cfg):
    def m():
        f = M.ajustar_prever_de(cfg["hp"], cfg["features"], cfg["over"])
        t0 = time.time()
        r = comum.medir_placar(X_tr, y_tr, f, threads=THREADS)
        r.update(arvores=float(np.median(f.arvores)), tempo_s=time.time() - t0)
        return r
    return m


def garantir(rotulo, cfg, contra=None):
    linha, medida = comum.garantir_placar(PASTA, M.MODELO, rotulo, cfg, medir(cfg), y_tr, ASS, contra)
    return linha


def mostrar(linhas, campos):
    rows = []
    for r in linhas:
        row = dict(rotulo=r["rotulo"], ks_kfold=r["ks_kfold"], ep=r["ep"], ks_oof=r["ks_oof"], auc_oof=r["auc_oof"],
                   mse_treino=r.get("mse_treino"), mse_val=r["mse_val"], ce_val=r["ce_val"], arvores=r.get("arvores"),
                   minutos=(r.get("tempo_s") or 0) / 60)
        for c in campos:
            v = r.get(c)
            row[c] = "" if not v else f"{v['veredito']} {v['delta']:+.4f} [{v['ic95'][0]:+.4f}, {v['ic95'][1]:+.4f}]"
        rows.append(row)
    display(pd.DataFrame(rows).round(4))

5282 clientes no treino | assinatura do treino 4c2839956f33bab5 | KS conferido contra scipy


## 1. Trial 0

Os padrões do `GradientBoostingClassifier`: `log_loss`, taxa 0,1, profundidade 3, folha mínima 1, sem amostragem de linhas nem de colunas, com o oversampling do grupo.
O número de árvores sai da parada, como no XGBoost, e não do padrão de 100.

In [4]:
T0 = (M.MODELO, "trial0")
CFG0 = config(M.PADRAO)
linha_t0 = garantir("trial0", CFG0)
mostrar([linha_t0], [])

,rotulo,ks_kfold,ep,ks_oof,auc_oof,mse_treino,mse_val,ce_val,arvores,minutos
0,trial0,0.5346,0.0065,0.5293,0.8439,0.1324,0.1595,0.4778,100.0,0.172


## 2. Busca no Optuna

Cada trial vale o KS médio das 5 dobras oficiais (semente 42), que não são as do placar.
O corte acontece depois da segunda dobra, como nos outros modelos.
A loss é um dos hiperparâmetros: `log_loss` (a "deviance" do slide 28) ou `exponential`, as duas que o scikit-learn oferece.

In [5]:
optuna.logging.set_verbosity(optuna.logging.WARNING)
t0 = time.time()
st = comum.estudo_placar(M.MODELO, N_TRIALS, M.sugerir, M.ajustar_prever_de, M.PADRAO, X_tr, y_tr, pasta=PASTA_ESTUDO, threads=THREADS)
ok = [t for t in st.trials if t.state.name == "COMPLETE"]
print(f"{len(ok)} completos, {len(st.trials) - len(ok)} podados | {(time.time() - t0) / 60:.1f} min nesta execução")
print(f"KS trial 0 = {st.trials[0].value:.4f} -> melhor = {st.best_value:.4f} (trial {st.best_trial.number})")
print("melhor:", st.best_params)
resumo = comum.sumario_estudo(st, "loss", ("log_loss", "exponential"))
print("por loss:", json.dumps(resumo["por_perda"], indent=1))
print("importâncias:", {k: round(v, 3) for k, v in resumo["importancias"].items()})
df = st.trials_dataframe()
display(df[df.state == "COMPLETE"].sort_values("value", ascending=False).head(10)[["number", "value"] + [c for c in df if c.startswith("params_")]].round(4))

86 completos, 14 podados | 0.0 min nesta execução
KS trial 0 = 0.5222 -> melhor = 0.5519 (trial 74)
melhor: {'loss': 'exponential', 'learning_rate': 0.009657826468194172, 'max_depth': 1, 'min_samples_leaf': 2, 'subsample': 0.7761257374899942, 'max_features': 0.3844108491531383, 'min_impurity_decrease': 0.0006083444832101401}


por loss: {
 "log_loss": {
  "n": 1,
  "melhor": 0.5221802525170467,
  "mediana": 0.5221802525170467,
  "media_top5": 0.5221802525170467
 },
 "exponential": {
  "n": 85,
  "melhor": 0.5519280499373685,
  "mediana": 0.5483738908484845,
  "media_top5": 0.5513311809810324
 }
}
importâncias: {'learning_rate': 0.387, 'subsample': 0.222, 'max_features': 0.132, 'max_depth': 0.108, 'min_impurity_decrease': 0.093, 'min_samples_leaf': 0.052, 'loss': 0.007}


,number,value,params_learning_rate,params_loss,params_max_depth,params_max_features,params_min_impurity_decrease,params_min_samples_leaf,params_subsample
74,74,0.5519,0.0097,exponential,1,0.3844,0.0006,2,0.7761
73,73,0.5516,0.0089,exponential,1,0.3825,0.0004,1,0.7499
87,87,0.5515,0.0127,exponential,1,0.3719,0.0002,1,0.7203
57,57,0.5509,0.0097,exponential,1,0.3924,0.0013,1,0.7517
91,91,0.5508,0.0099,exponential,1,0.3044,0.0003,2,0.7023
80,80,0.5507,0.0093,exponential,1,0.3261,0.0003,1,0.7070
90,90,0.5506,0.0083,exponential,1,0.3084,0.0007,2,0.7533
94,94,0.5506,0.0090,exponential,1,0.3117,0.0005,3,0.7536
83,83,0.5506,0.0113,exponential,1,0.3153,0.0006,2,0.7173
70,70,0.5504,0.0095,exponential,1,0.3512,0.0006,2,0.6922


## 3. O melhor trial no placar

O KS do melhor trial nas dobras oficiais é otimista, porque foi escolhido nelas.
Aqui ele é medido nas 15 divisões do placar, contra o trial 0.

In [6]:
HP_OPTUNA = M.sugerir(optuna.trial.FixedTrial(st.best_params))
linha_opt = garantir("optuna", config(HP_OPTUNA), {"vs_trial0": T0})
mostrar([linha_t0, linha_opt], ["vs_trial0", "vs_xgboost"])

,rotulo,ks_kfold,ep,ks_oof,auc_oof,mse_treino,mse_val,ce_val,arvores,minutos,vs_trial0,vs_xgboost
0,trial0,0.5346,0.0065,0.5293,0.8439,0.1324,0.1595,0.4778,100.0,0.1720,,"empatou -0.0098 [-0.0213, +0.0033]"
1,optuna,0.5457,0.0068,0.5389,0.8469,0.1558,0.1641,0.4883,1298.0,1.6517,"empatou +0.0096 [-0.0023, +0.0219]","empatou -0.0001 [-0.0043, +0.0050]"


## 4. Dados: oversampling e features

As duas perguntas do `docs/GUIA_DADOS.md`, com os hiperparâmetros do melhor trial parados.
No XGBoost, nenhuma das duas mudou o KS no placar.

In [7]:
TRES = ("cliente_novo", "delta_preco", "tenure_1")
OPT = (M.MODELO, "optuna")
variantes = {"sem_over": config(HP_OPTUNA, over=False), "tres_features": config(HP_OPTUNA, features=TRES)}
linhas_var = {rot: garantir(rot, cfg, {"vs_optuna": OPT, "vs_trial0": T0}) for rot, cfg in variantes.items()}
mostrar([linha_opt, *linhas_var.values()], ["vs_optuna", "vs_trial0"])

,rotulo,ks_kfold,ep,ks_oof,auc_oof,mse_treino,mse_val,ce_val,arvores,minutos,vs_optuna,vs_trial0
0,optuna,0.5457,0.0068,0.5389,0.8469,0.1558,0.1641,0.4883,1298.0,1.6517,,"empatou +0.0096 [-0.0023, +0.0219]"
1,sem_over,0.5496,0.0075,0.5426,0.8474,0.1324,0.1343,0.4145,1616.0,1.8965,"empatou +0.0037 [-0.0031, +0.0082]","ganhou +0.0133 [+0.0008, +0.0232]"
2,tres_features,0.5488,0.0064,0.5414,0.8460,0.1554,0.1630,0.4871,1349.0,2.0142,"empatou +0.0025 [-0.0071, +0.0112]","empatou +0.0121 [-0.0006, +0.0237]"


## 5. Final

Fica o melhor trial, a menos que uma variante tenha **ganhado** dele: empate não troca.

In [8]:
ganharam = {rot: l for rot, l in linhas_var.items() if (l.get("vs_optuna") or {}).get("veredito") == "ganhou"}
if ganharam:
    rot_final = max(ganharam, key=lambda r: ganharam[r]["ks_oof"])
    CFG_FINAL, motivo = variantes[rot_final], f"{rot_final} ganhou do melhor trial no placar"
else:
    rot_final, CFG_FINAL = "optuna", config(HP_OPTUNA)
    motivo = "melhor trial do Optuna; nenhuma variante de dados ganhou dele no placar"
linha_final = garantir("final", CFG_FINAL, {"vs_trial0": T0})
comum.adotar_placar(PASTA, M.MODELO, "final", motivo)
print("final =", rot_final, "|", motivo)
print(json.dumps(CFG_FINAL, indent=1))
mostrar([linha_final], ["vs_trial0", "vs_xgboost"])

final = optuna | melhor trial do Optuna; nenhuma variante de dados ganhou dele no placar
{
 "hp": {
  "loss": "exponential",
  "learning_rate": 0.009657826468194172,
  "max_depth": 1,
  "min_samples_leaf": 2,
  "subsample": 0.7761257374899942,
  "max_features": 0.3844108491531383,
  "min_impurity_decrease": 0.0006083444832101401
 },
 "features": [],
 "over": true
}


,rotulo,ks_kfold,ep,ks_oof,auc_oof,mse_treino,mse_val,ce_val,arvores,minutos,vs_trial0,vs_xgboost
0,final,0.5457,0.0068,0.5389,0.8469,0.1558,0.1641,0.4883,1298.0,1.9159,"empatou +0.0096 [-0.0023, +0.0219]","empatou -0.0001 [-0.0043, +0.0050]"


## 6. Teste

Só com `REVELAR_TESTE = True`, uma vez, com a configuração final já congelada.
O limiar das métricas de classificação vem da OOF do placar, nunca do teste.

In [9]:
if REVELAR_TESTE:
    fin = M.final(CFG_FINAL["hp"], X_tr, y_tr, X_te, N_SEMENTES, CFG_FINAL["features"], CFG_FINAL["over"])
    comum.registrar_teste_placar(PASTA, M.MODELO, "final", y_te.values, fin["p_sementes"])
    oof = np.load(os.path.join(PASTA, linha_final["oof"])).mean(0)
    limiar = comum.limiar_ks(y_tr.values, oof)
    p = np.mean(fin["p_sementes"], 0)
    m = comum.metricas(y_te.values, p, limiar)
    m.update(ks_ic95=comum.bootstrap_ks(y_te.values, p), ks_por_semente=[comum.ks(y_te.values, np.asarray(x)) for x in fin["p_sementes"]],
             arvores=fin["arvores"])
    json.dump(dict(config=CFG_FINAL, teste=m, **fin, y_teste=[int(v) for v in y_te]), open(os.path.join(PASTA_ESTUDO, "teste.json"), "w"), indent=1)
    print(json.dumps({k: v for k, v in m.items() if k != "confusao"}, indent=1)); print("confusão:", m["confusao"])
else:
    print("Teste não revelado (REVELAR_TESTE = False).")

{
 "ks": 0.5422754998361736,
 "auroc": 0.845941902836018,
 "ce": 0.48714314895467453,
 "mse": 0.1647401240223911,
 "precision": 0.5206611570247934,
 "recall": 0.8094218415417559,
 "f1": 0.633696563285834,
 "limiar": 0.5199021697044373,
 "ks_ic95": [
  0.5017165846753973,
  0.5837386335854345
 ],
 "ks_por_semente": [
  0.5369304548418165,
  0.5439981598482868,
  0.5419693594881996,
  0.544529354722339,
  0.5406653670870993
 ],
 "arvores": [
  1191,
  1389,
  1668,
  1220,
  1216
 ]
}
confusão: [[946, 348], [89, 378]]


In [10]:
display(comum.tabela_placar(PASTA).query("modelo == @M.MODELO").round(4))

,modelo,rotulo,ks_kfold,ep,ks_oof,auc_oof,mse_val,mse_treino,vs_trial0,vs_xgboost,ks_teste,auc_teste,vs_referencia,vs_adotado,vs_optuna,vs_trial0_float32
53,gradient_boosting,trial0,0.5346,0.0065,0.5293,0.8439,0.1595,0.1324,NaN,"empatou (-0.0098 [-0.0213, +0.0033])",NaN,NaN,NaN,NaN,NaN,NaN
54,gradient_boosting,optuna,0.5457,0.0068,0.5389,0.8469,0.1641,0.1558,"empatou (+0.0096 [-0.0023, +0.0219])","empatou (-0.0001 [-0.0043, +0.0050])",NaN,NaN,NaN,NaN,NaN,NaN
55,gradient_boosting,sem_over,0.5496,0.0075,0.5426,0.8474,0.1343,0.1324,"ganhou (+0.0133 [+0.0008, +0.0232])","empatou (+0.0036 [-0.0037, +0.0096])",NaN,NaN,NaN,NaN,"empatou (+0.0037 [-0.0031, +0.0082])",NaN
56,gradient_boosting,tres_features,0.5488,0.0064,0.5414,0.8460,0.1630,0.1554,"empatou (+0.0121 [-0.0006, +0.0237])","empatou (+0.0024 [-0.0070, +0.0117])",NaN,NaN,NaN,NaN,"empatou (+0.0025 [-0.0071, +0.0112])",NaN
57,gradient_boosting,final,0.5457,0.0068,0.5389,0.8469,0.1641,0.1558,"empatou (+0.0096 [-0.0023, +0.0219])","empatou (-0.0001 [-0.0043, +0.0050])",0.5423,0.8459,NaN,NaN,NaN,NaN
